## setting up

In [34]:
!pip install linearmodels

In [35]:
#package
import os
import pandas as pd
import numpy as np
import statsmodels.api as sm
from linearmodels.iv import IV2SLS, IVGMM

In [36]:
# environmental info
import sys
import pandas as pd
import numpy as np
import matplotlib
import statsmodels
import linearmodels

# Print package versions
print(f"Python:       {sys.version.split()[0]}")
print(f"pandas:       {pd.__version__}")
print(f"numpy:        {np.__version__}")
print(f"matplotlib:   {matplotlib.__version__}")
print(f"statsmodels:  {statsmodels.__version__}")
print(f"linearmodels: {linearmodels.__version__}")

Python:       3.13.18
pandas:       2.2.3
numpy:        2.4.6
matplotlib:   3.10.0
statsmodels:  0.15.0
linearmodels: 7.1


In [37]:
#set current directrory
# os.chdir("")

In [38]:
df_card = pd.read_stata("data/CARD.DTA")

## A

### Data preparation

In [39]:

# Set random seed
np.random.seed(11111)

# Sample size
n = 1000

# Generate independent random variables
epsilon = np.random.normal(0, 1, n)
x2 = np.random.normal(1, 1, n)
z = np.random.normal(1, np.sqrt(10), n)
u = np.random.uniform(-5, 5, n)

# Create DataFrame
df_sim = pd.DataFrame({
    "epsilon": epsilon,
    "x2": x2,
    "z": z,
    "u": u
})

# Generate additional variables
#x
df_sim["x1"] = 1
df_sim["x3"] = 0.05 * df_sim["z"] + df_sim["epsilon"]
df_sim["x4"] = 0.0001 * df_sim["z"] + df_sim["epsilon"]
#y
df_sim["y1"] = df_sim["x1"] + 2*df_sim["x2"] + 3*df_sim["x3"] + df_sim["epsilon"]
df_sim["y2"] = df_sim["x1"] + 2*df_sim["x2"] + 3*df_sim["x3"] + 4*df_sim["z"] + df_sim["u"]
df_sim["y3"] = df_sim["x1"] + 2*df_sim["x2"] + 3*df_sim["x4"] + df_sim["epsilon"]


### A1. Endogeneity and IV.

In [40]:
# OLS with robust standard errors (HC1)
X_ols_a1 = sm.add_constant(df_sim[["x2", "x3"]], has_constant="add")
y_ols_a1 = df_sim["y1"]

model_ols_a1 = sm.OLS(y_ols_a1, X_ols_a1).fit(cov_type="HC1")

print(model_ols_a1.summary())

                            OLS Regression Results                            
Dep. Variable:                     y1   R-squared:                       0.999
Model:                            OLS   Adj. R-squared:                  0.999
Method:                 Least Squares   F-statistic:                 3.678e+05
Date:                Fri, 13 Nov 2026   Prob (F-statistic):               0.00
Time:                        06:33:25   Log-Likelihood:                 429.01
No. Observations:                1000   AIC:                            -852.0
Df Residuals:                     997   BIC:                            -837.3
Df Model:                           2                                         
Covariance Type:                  HC1                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.9488      0.008    124.076      0.0

In [41]:
# 2SLS with robust standard errors (HC1)
model_iv_a1 = IV2SLS(
    dependent=df_sim["y1"],
    exog=df_sim[["x1", "x2"]],
    endog=df_sim["x3"],
    instruments=df_sim["z"]
).fit(cov_type="robust", debiased=True)

# First stage
print(model_iv_a1.first_stage)

    First Stage Estimation Results    
                                    x3
--------------------------------------
R-squared                       0.0205
Partial R-squared               0.0175
Shea's R-squared                0.0175
Partial F-statistic             16.053
P-value (Partial F-stat)     6.159e-05
Partial F-stat Distn           chi2(1)
========================== ===========
x1                              0.0813
                              (1.7037)
x2                             -0.0545
                             (-1.6387)
z                               0.0426
                              (4.0066)
--------------------------------------

T-stats reported in parentheses
T-stats use same covariance type as original model


In [42]:
# Second stage
print(model_iv_a1.summary)

                          IV-2SLS Estimation Summary                          
Dep. Variable:                     y1   R-squared:                      0.9276
Estimator:                    IV-2SLS   Adj. R-squared:                 0.9275
No. Observations:                1000   F-statistic:                    1117.9
Date:                Fri, Nov 13 2026   P-value (F-stat)                0.0000
Time:                        06:33:25   Distribution:                 F(2,997)
Cov. Estimator:                robust                                         
                                                                              
                             Parameter Estimates                              
            Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
------------------------------------------------------------------------------
x1             1.0954     0.0646     16.957     0.0000      0.9687      1.2222
x2             1.9361     0.0413     46.845     0.00

In [43]:
#create latex table
# First-stage results
fs_a1 = model_iv_a1.first_stage.individual["x3"]
diag_a1 = model_iv_a1.first_stage.diagnostics.loc["x3"]

# Create LaTeX table
table_a1 = pd.DataFrame({
    "OLS": [
        f"{model_ols_a1.params[v]:.4f}\n({model_ols_a1.bse[v]:.4f})"
        for v in ["const", "x2", "x3"]
    ],
    "2SLS": [
        f"{model_iv_a1.params[v]:.4f}\n({model_iv_a1.std_errors[v]:.4f})"
        for v in ["x1", "x2", "x3"]
    ]
}, index=["Constant", "x2", "x3"])

table_a1.loc["N"] = [
    f"{model_ols_a1.nobs:.0f}",
    f"{model_iv_a1.nobs:.0f}"
]

table_a1.loc["R-squared"] = [
    f"{model_ols_a1.rsquared:.4f}",
    f"{model_iv_a1.rsquared:.4f}"
]

# First-stage results below the main estimates
table_a1.loc["First Stage: x3"] = ["", ""]
table_a1.loc["z coefficient"] = [
    "",
    f"{fs_a1.params['z']:.4f}\n({fs_a1.std_errors['z']:.4f})"
]
table_a1.loc["Partial F-statistic"] = [
    "", f"{diag_a1['f.stat']:.4f}"
]
table_a1.loc["P-value"] = [
    "", f"{diag_a1['f.pval']:.4f}"
]
table_a1.loc["Partial R-squared"] = [
    "", f"{diag_a1['partial.rsquared']:.4f}"
]

# Export to LaTeX
latex_a1 = table_a1.to_latex(
    escape=False,
    caption="OLS and 2SLS Estimation Results",
    label="tab:a1",
    column_format="lcc"
)

print(latex_a1)


\begin{table}
\caption{OLS and 2SLS Estimation Results}
\label{tab:a1}
\begin{tabular}{lcc}
\toprule
 & OLS & 2SLS \\
\midrule
Constant & 0.9488
(0.0076) & 1.0954
(0.0646) \\
x2 & 2.0030
(0.0055) & 1.9361
(0.0413) \\
x3 & 3.9795
(0.0050) & 2.8265
(0.2929) \\
N & 1000 & 1000 \\
R-squared & 0.9987 & 0.9276 \\
First Stage: x3 &  &  \\
z coefficient &  & 0.0426
(0.0106) \\
Partial F-statistic &  & 16.0531 \\
P-value &  & 0.0001 \\
Partial R-squared &  & 0.0175 \\
\bottomrule
\end{tabular}
\end{table}



### A2. A direct effect of z.

In [44]:
# A2: OLS with robust standard errors (HC1)
X_ols_a2 = sm.add_constant(df_sim[["x2", "x3", "z"]], has_constant="add")
y_ols_a2 = df_sim["y2"]

model_ols_a2 = sm.OLS(y_ols_a2, X_ols_a2).fit(cov_type="HC1")

print(model_ols_a2.summary())

                            OLS Regression Results                            
Dep. Variable:                     y2   R-squared:                       0.956
Model:                            OLS   Adj. R-squared:                  0.956
Method:                 Least Squares   F-statistic:                     7529.
Date:                Fri, 13 Nov 2026   Prob (F-statistic):               0.00
Time:                        06:33:25   Log-Likelihood:                -2479.9
No. Observations:                1000   AIC:                             4968.
Df Residuals:                     996   BIC:                             4987.
Df Model:                           3                                         
Covariance Type:                  HC1                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          1.0949      0.137      7.990      0.0

In [45]:
# A2: 2SLS with robust standard errors
model_iv_a2 = IV2SLS(
    dependent=df_sim["y2"],
    exog=df_sim[["x1", "x2"]],
    endog=df_sim["x3"],
    instruments=df_sim["z"]
).fit(cov_type="robust", debiased=True)

print(model_iv_a2.summary)

# First stage
print(model_iv_a2.first_stage)

                          IV-2SLS Estimation Summary                          
Dep. Variable:                     y2   R-squared:                     -46.703
Estimator:                    IV-2SLS   Adj. R-squared:                -46.798
No. Observations:                1000   F-statistic:                    8.8321
Date:                Fri, Nov 13 2026   P-value (F-stat)                0.0002
Time:                        06:33:25   Distribution:                 F(2,997)
Cov. Estimator:                robust                                         
                                                                              
                             Parameter Estimates                              
            Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
------------------------------------------------------------------------------
x1            -6.5289     5.1770    -1.2611     0.2076     -16.688      3.6301
x2             7.1475     3.3086     2.1603     0.03

In [46]:
# First-stage results
fs_a2 = model_iv_a2.first_stage.individual["x3"]
diag_a2 = model_iv_a2.first_stage.diagnostics.loc["x3"]

# Create LaTeX table
table_a2 = pd.DataFrame({
    "OLS": [
        f"{model_ols_a2.params[v]:.4f}\n({model_ols_a2.bse[v]:.4f})"
        for v in ["const", "x2", "x3", "z"]
    ],
    "2SLS": [
        f"{model_iv_a2.params[v]:.4f}\n({model_iv_a2.std_errors[v]:.4f})"
        if v != "z" else ""
        for v in ["x1", "x2", "x3", "z"]
    ]
}, index=["Constant", "x2", "x3", "z"])

table_a2.loc["N"] = [
    f"{model_ols_a2.nobs:.0f}",
    f"{model_iv_a2.nobs:.0f}"
]

table_a2.loc["R-squared"] = [
    f"{model_ols_a2.rsquared:.4f}",
    f"{model_iv_a2.rsquared:.4f}"
]

# First-stage results
table_a2.loc["First Stage: x3"] = ["", ""]

table_a2.loc["Constant (First Stage)"] = [
    "",
    f"{fs_a2.params['x1']:.4f}\n({fs_a2.std_errors['x1']:.4f})"
]

table_a2.loc["x2 (First Stage)"] = [
    "",
    f"{fs_a2.params['x2']:.4f}\n({fs_a2.std_errors['x2']:.4f})"
]

table_a2.loc["z (First Stage)"] = [
    "",
    f"{fs_a2.params['z']:.4f}\n({fs_a2.std_errors['z']:.4f})"
]

table_a2.loc["Partial F-statistic"] = [
    "", f"{diag_a2['f.stat']:.4f}"
]

table_a2.loc["P-value"] = [
    "", f"{diag_a2['f.pval']:.4f}"
]

table_a2.loc["Partial R-squared"] = [
    "", f"{diag_a2['partial.rsquared']:.4f}"
]

# Export to LaTeX
latex_a2 = table_a2.to_latex(
    escape=False,
    caption="OLS and 2SLS Estimation Results (A2)",
    label="tab:a2",
    column_format="lcc"
)

print(latex_a2)


\begin{table}
\caption{OLS and 2SLS Estimation Results (A2)}
\label{tab:a2}
\begin{tabular}{lcc}
\toprule
 & OLS & 2SLS \\
\midrule
Constant & 1.0949
(0.1370) & -6.5289
(5.1770) \\
x2 & 2.0434
(0.0912) & 7.1475
(3.3086) \\
x3 & 2.8298
(0.0888) & 96.5606
(23.4218) \\
z & 3.9938
(0.0285) &  \\
N & 1000 & 1000 \\
R-squared & 0.9561 & -46.7027 \\
First Stage: x3 &  &  \\
Constant (First Stage) &  & 0.0813
(0.0477) \\
x2 (First Stage) &  & -0.0545
(0.0332) \\
z (First Stage) &  & 0.0426
(0.0106) \\
Partial F-statistic &  & 16.0531 \\
P-value &  & 0.0001 \\
Partial R-squared &  & 0.0175 \\
\bottomrule
\end{tabular}
\end{table}



### A3. A weak first stage.

In [47]:
# A3: 2SLS with robust standard errors
model_iv_a3 = IV2SLS(
    dependent=df_sim["y3"],
    exog=df_sim[["x1", "x2"]],
    endog=df_sim["x4"],
    instruments=df_sim["z"]
).fit(cov_type="robust", debiased=True)


print(model_iv_a3.summary)


                          IV-2SLS Estimation Summary                          
Dep. Variable:                     y3   R-squared:                      1.0000
Estimator:                    IV-2SLS   Adj. R-squared:                 1.0000
No. Observations:                1000   F-statistic:                 7.645e+06
Date:                Fri, Nov 13 2026   P-value (F-stat)                0.0000
Time:                        06:33:25   Distribution:                 F(2,997)
Cov. Estimator:                robust                                         
                                                                              
                             Parameter Estimates                              
            Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
------------------------------------------------------------------------------
x1             0.9989     0.0016     607.54     0.0000      0.9957      1.0021
x2             2.0007     0.0012     1662.8     0.00

In [48]:
# First stage
print(model_iv_a3.first_stage)

    First Stage Estimation Results    
                                    x4
--------------------------------------
R-squared                       0.0032
Partial R-squared               0.0005
Shea's R-squared                0.0005
Partial F-statistic             0.4700
P-value (Partial F-stat)        0.4930
Partial F-stat Distn           chi2(1)
========================== ===========
x1                              0.0813
                              (1.7037)
x2                             -0.0545
                             (-1.6387)
z                              -0.0073
                             (-0.6856)
--------------------------------------

T-stats reported in parentheses
T-stats use same covariance type as original model


In [49]:

# First-stage results
fs_a3 = model_iv_a3.first_stage.individual["x4"]
diag_a3 = model_iv_a3.first_stage.diagnostics.loc["x4"]

# Create LaTeX table
table_a3 = pd.DataFrame({
    "2SLS": [
        f"{model_iv_a3.params[v]:.4f} ({model_iv_a3.std_errors[v]:.4f})"
        for v in ["x1", "x2", "x4"]
    ]
}, index=["Constant", "x2", "x4"])

table_a3.loc["N"] = f"{model_iv_a3.nobs:.0f}"
table_a3.loc["R-squared"] = f"{model_iv_a3.rsquared:.4f}"

# First-stage results
table_a3.loc["First Stage: x4"] = ""

for v, name in [("x1", "Constant"), ("x2", "x2"), ("z", "z")]:
    table_a3.loc[f"First Stage: {name}"] = (
        f"{fs_a3.params[v]:.4f} ({fs_a3.std_errors[v]:.4f})"
    )

table_a3.loc["Partial F-statistic"] = f"{diag_a3['f.stat']:.4f}"
table_a3.loc["P-value"] = f"{diag_a3['f.pval']:.4f}"
table_a3.loc["Partial R-squared"] = f"{diag_a3['partial.rsquared']:.4f}"

# Export to LaTeX
latex_a3 = table_a3.to_latex(
    escape=False,
    caption="2SLS Estimation Results (A3)",
    label="tab:a3",
    column_format="lc"
)

print(latex_a3)



\begin{table}
\caption{2SLS Estimation Results (A3)}
\label{tab:a3}
\begin{tabular}{lc}
\toprule
 & 2SLS \\
\midrule
Constant & 0.9989 (0.0016) \\
x2 & 2.0007 (0.0012) \\
x4 & 4.0137 (0.0200) \\
N & 1000 \\
R-squared & 1.0000 \\
First Stage: x4 &  \\
First Stage: Constant & 0.0813 (0.0477) \\
First Stage: x2 & -0.0545 (0.0332) \\
First Stage: z & -0.0073 (0.0106) \\
Partial F-statistic & 0.4700 \\
P-value & 0.4930 \\
Partial R-squared & 0.0005 \\
\bottomrule
\end{tabular}
\end{table}



## B

#### Data preparation

In [50]:
df_card.columns

Index(['id', 'nearc2', 'nearc4', 'educ', 'age', 'fatheduc', 'motheduc',
       'weight', 'momdad14', 'sinmom14', 'step14', 'reg661', 'reg662',
       'reg663', 'reg664', 'reg665', 'reg666', 'reg667', 'reg668', 'reg669',
       'south66', 'black', 'smsa', 'south', 'smsa66', 'wage', 'enroll', 'KWW',
       'IQ', 'married', 'libcrd14'],
      dtype='object')

In [51]:
#create variables
df_card["lwage"] = np.log(df_card["wage"].where(df_card["wage"] > 0))
df_card["exper"] = df_card["age"] - df_card["educ"] - 6
df_card["expersq"] = df_card["exper"] ** 2

In [52]:
#control variables
controls_card = [
    "exper", "expersq", "black", "south", "smsa", "smsa66",
    "reg662", "reg663", "reg664", "reg665",
    "reg666", "reg667", "reg668", "reg669"
]
#all variables
vars_card = ["lwage", "educ", "nearc4", "nearc2"] + controls_card
#dropna
df_card_reg = df_card[vars_card].dropna().copy()
#setting const
df_card_reg["const"] = 1

### Estimation

In [53]:
#model1

model_iv_card_1 = IV2SLS(
    dependent=df_card_reg["lwage"],
    exog=df_card_reg[["const"] + controls_card],
    endog=df_card_reg["educ"],
    instruments=df_card_reg[["nearc4"]]
).fit(cov_type="robust", debiased=False)

print(model_iv_card_1.summary)
print(model_iv_card_1.first_stage)


                          IV-2SLS Estimation Summary                          
Dep. Variable:                  lwage   R-squared:                      0.2230
Estimator:                    IV-2SLS   Adj. R-squared:                 0.2191
No. Observations:                3010   F-statistic:                    765.74
Date:                Fri, Nov 13 2026   P-value (F-stat)                0.0000
Time:                        06:33:25   Distribution:                 chi2(15)
Cov. Estimator:                robust                                         
                                                                              
                             Parameter Estimates                              
            Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
------------------------------------------------------------------------------
const          3.7588     0.9311     4.0368     0.0001      1.9338      5.5838
exper          0.0653     0.0222     2.9347     0.00

In [54]:
#model2
model_gmm_card_2 = IVGMM(
    dependent=df_card_reg["lwage"],
    exog=df_card_reg[["const"] + controls_card],
    endog=df_card_reg["educ"],
    instruments=df_card_reg[["nearc4"]],
    weight_type="robust"
).fit(iter_limit=2, cov_type="robust", debiased=False)

print(model_gmm_card_2.summary)
print(model_gmm_card_2.first_stage)


                          IV-GMM Estimation Summary                           
Dep. Variable:                  lwage   R-squared:                      0.2230
Estimator:                     IV-GMM   Adj. R-squared:                 0.2191
No. Observations:                3010   F-statistic:                    765.74
Date:                Fri, Nov 13 2026   P-value (F-stat)                0.0000
Time:                        06:33:26   Distribution:                 chi2(15)
Cov. Estimator:                robust                                         
                                                                              
                             Parameter Estimates                              
            Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
------------------------------------------------------------------------------
const          3.7588     0.9311     4.0368     0.0001      1.9338      5.5838
exper          0.0653     0.0222     2.9347     0.00

In [55]:
#model 3
model_iv_card_3 = IV2SLS(
    dependent=df_card_reg["lwage"],
    exog=df_card_reg[["const"] + controls_card],
    endog=df_card_reg["educ"],
    instruments=df_card_reg[["nearc4", "nearc2"]]
).fit(cov_type="robust", debiased=False)

print(model_iv_card_3.summary)
print(model_iv_card_3.first_stage)


                          IV-2SLS Estimation Summary                          
Dep. Variable:                  lwage   R-squared:                      0.1548
Estimator:                    IV-2SLS   Adj. R-squared:                 0.1506
No. Observations:                3010   F-statistic:                    708.55
Date:                Fri, Nov 13 2026   P-value (F-stat)                0.0000
Time:                        06:33:26   Distribution:                 chi2(15)
Cov. Estimator:                robust                                         
                                                                              
                             Parameter Estimates                              
            Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
------------------------------------------------------------------------------
const          3.3365     0.9057     3.6838     0.0002      1.5613      5.1117
exper          0.0754     0.0217     3.4816     0.00

In [56]:
#model4

model_gmm_card_4 = IVGMM(
    dependent=df_card_reg["lwage"],
    exog=df_card_reg[["const"] + controls_card],
    endog=df_card_reg["educ"],
    instruments=df_card_reg[["nearc4", "nearc2"]],
    weight_type="robust"
).fit(iter_limit=2, cov_type="robust", debiased=False)

print(model_gmm_card_4.summary)
print(model_gmm_card_4.first_stage)
print(model_gmm_card_4.j_stat)


                          IV-GMM Estimation Summary                           
Dep. Variable:                  lwage   R-squared:                      0.1604
Estimator:                     IV-GMM   Adj. R-squared:                 0.1562
No. Observations:                3010   F-statistic:                    714.93
Date:                Fri, Nov 13 2026   P-value (F-stat)                0.0000
Time:                        06:33:27   Distribution:                 chi2(15)
Cov. Estimator:                robust                                         
                                                                              
                             Parameter Estimates                              
            Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
------------------------------------------------------------------------------
const          3.3655     0.9023     3.7298     0.0002      1.5970      5.1340
exper          0.0746     0.0216     3.4609     0.00

In [57]:
#create table

models_card = [
    model_iv_card_1,
    model_gmm_card_2,
    model_iv_card_3,
    model_gmm_card_4
]

labels_card = {
    "const": "Constant",
    "educ": "Years of schooling, 1976",
    "nearc4": "=1 if grew up near a four-year college, 1966",
    "nearc2": "=1 if grew up near a two-year college, 1966",
    "exper": "Potential experience (age - educ - 6), 1976",
    "expersq": "Potential experience squared",
    "black": "=1 if black",
    "south": "=1 if lived in the South, 1976",
    "smsa": "=1 if lived in an SMSA, 1976",
    "smsa66": "=1 if lived in an SMSA, 1966",
    **{f"reg66{i}": f"=1 if lived in region {i}, 1966"
       for i in range(2, 10)}
}

def coef_se(model, var):
    if var not in model.params.index:
        return ""
    return (
        f"{model.params[var]:.4f} "
        f"({model.std_errors[var]:.4f})"
    )

def fmt_p(p):
    return "<0.0001" if p < 0.0001 else f"{p:.4f}"

table_card = pd.DataFrame(
    columns=["(1) 2SLS", "(2) GMM", "(3) 2SLS", "(4) GMM"]
)

# Structural equation
table_card.loc["Structural equation"] = [""] * 4

for v in ["const", "educ"] + controls_card:
    table_card.loc[labels_card[v]] = [
        coef_se(m, v) for m in models_card
    ]

table_card.loc["Observations"] = [
    f"{m.nobs:.0f}" for m in models_card
]

table_card.loc["R-squared"] = [
    f"{m.rsquared:.4f}" for m in models_card
]

# First stage
table_card.loc["First stage: Years of schooling, 1976"] = [""] * 4

fs_models_card = [
    m.first_stage.individual["educ"] for m in models_card
]

for v in ["const"] + controls_card + ["nearc4", "nearc2"]:
    table_card.loc["First stage: " + labels_card[v]] = [
        coef_se(fs, v) for fs in fs_models_card
    ]

# Robust first-stage F tests (HC1)
fstats_card = []
fpvals_card = []

for i, fs in enumerate(fs_models_card):
    excluded = ["nearc4"] if i < 2 else ["nearc4", "nearc2"]
    restriction = ", ".join(f"{v} = 0" for v in excluded)

    fs_robust = sm.OLS(
        df_card_reg["educ"],
        df_card_reg[["const"] + controls_card + excluded]
    ).fit(cov_type="HC1")

    test = fs_robust.f_test(restriction)
    fstats_card.append(float(test.fvalue))
    fpvals_card.append(float(test.pvalue))

table_card.loc["Robust first-stage F"] = [
    f"{v:.4f}" for v in fstats_card
]

table_card.loc["First-stage F p-value"] = [
    fmt_p(v) for v in fpvals_card
]

# Kleibergen-Paap rk LM
kp_card = [
    m.first_stage.kleibergen_paap for m in models_card
]

table_card.loc["Kleibergen-Paap rk LM"] = [
    f"{kp.stat:.4f}" for kp in kp_card
]

table_card.loc["KP rk LM p-value"] = [
    fmt_p(kp.pval) for kp in kp_card
]

# Hansen J: Model (4)
table_card.loc["Hansen J"] = [
    "", "", "", f"{model_gmm_card_4.j_stat.stat:.4f}"
]

table_card.loc["Hansen J p-value"] = [
    "", "", "", fmt_p(model_gmm_card_4.j_stat.pval)
]

# Export to LaTeX
latex_card = table_card.to_latex(
    escape=True,
    caption="Returns to Schooling: 2SLS and Two-Step GMM",
    label="tab:card_iv_gmm",
    column_format="lcccc"
)

with open("table_card.tex", "w", encoding="utf-8") as f:
    f.write(latex_card)

print(latex_card)

\begin{table}
\caption{Returns to Schooling: 2SLS and Two-Step GMM}
\label{tab:card_iv_gmm}
\begin{tabular}{lcccc}
\toprule
 & (1) 2SLS & (2) GMM & (3) 2SLS & (4) GMM \\
\midrule
Structural equation &  &  &  &  \\
Constant & 3.7588 (0.9311) & 3.7588 (0.9311) & 3.3365 (0.9057) & 3.3655 (0.9023) \\
Years of schooling, 1976 & 0.1347 (0.0550) & 0.1347 (0.0550) & 0.1597 (0.0535) & 0.1579 (0.0533) \\
Potential experience (age - educ - 6), 1976 & 0.0653 (0.0222) & 0.0653 (0.0222) & 0.0754 (0.0217) & 0.0746 (0.0216) \\
Potential experience squared & 0.0005 (0.0002) & 0.0005 (0.0002) & 0.0006 (0.0002) & 0.0006 (0.0002) \\
=1 if black & -0.1454 (0.0526) & -0.1454 (0.0526) & -0.1229 (0.0517) & -0.1251 (0.0515) \\
=1 if lived in the South, 1976 & -0.1493 (0.0294) & -0.1493 (0.0294) & -0.1478 (0.0306) & -0.1476 (0.0305) \\
=1 if lived in an SMSA, 1976 & 0.1167 (0.0311) & 0.1167 (0.0311) & 0.1061 (0.0315) & 0.1068 (0.0314) \\
=1 if lived in an SMSA, 1966 & 0.0182 (0.0206) & 0.0182 (0.0206) & 0.0150 

###